# 13 · Verifica: il device è fermo davvero?
*Blocco 3 · Certificati. Passo di verifica, con un ripristino in fondo.*

In [ ]:
# Carica la libreria e legge lo scenario (notebooks\_scenario.psd1): da qui in poi si usa $cfg
. ([scriptblock]::Create([IO.File]::ReadAllText((Get-ChildItem . , .\notebooks , .. -Filter _start.ps1 -ErrorAction SilentlyContinue | Select-Object -First 1).FullName)))

Fabric: valori dalla Eventstream creata dal 17a (417bd75b-4747-4eff-9d6e-d00ca5964001)

Scenario: rg-iot26-dev-uks-001 | namespace ADR adrns-iot-dev-uks | output in D:\Dev\2026\IoTHub\WPC2026\artifacts

In [ ]:
if (-not $cfg -or -not $out) { throw "Mancano le variabili dello scenario: esegui prima la cella di caricamento (la prima cella di codice) di questo notebook." }
$id = 'device-da-revocare'
"stato in ADR: " + (Get-AdrDeviceState -Name $id -Namespace $cfg.AdrName -ResourceGroup $cfg.Rg)

stato in ADR: Disabled

## Anche il certificato valido è rifiutato
Il certificato ottenuto dopo la revoca (notebook 11) è valido, ma il device è disabilitato.

In [ ]:
$nuovo = Open-ScenarioDevice -Config $cfg -DeviceId $id -Notebook '11-device-nuova-registrazione'
Show-HubAcceptance -Label 'certificato valido, device disabilitato' -Hub $nuovo.AssignedHub -DeviceId $id -StateDir $nuovo.Dir

certificato valido, device disabilitato              HTTP 401  (rifiutato)

## Cosa abbiamo visto
- La revoca invalida i **certificati**; la disabilitazione ferma il **device**.
- Con il device disabilitato, un certificato valido viene rifiutato e una nuova registrazione fallisce.

## Ripristino
Riabilita il device, così i notebook 09-13 si possono rilanciare.

In [ ]:
"stato in ADR: " + (Set-AdrDeviceEnablement -Name $id -Namespace $cfg.AdrName -ResourceGroup $cfg.Rg -State Enabled)

stato in ADR: Enabled

## Dopo la riabilitazione
In ADR il device è di nuovo abilitato, ma **l'Hub non torna subito a fidarsi dei certificati di prima**. Nelle prove del 2026-11-24 il certificato valido è rimasto rifiutato (401) per minuti, in un caso per più di 18 minuti; in un altro caso è tornato accettato dopo circa 90 secondi. Il tempo non è prevedibile.
Cosa dice l'Hub subito dopo la riabilitazione, del certificato **valido** (notebook 11) e di quello **revocato** (notebook 09)?


In [ ]:
$prima = Open-ScenarioDevice -Config $cfg -DeviceId $id -Notebook '09-device-revoca'
Show-HubAcceptance -Label 'certificato valido, device riabilitato' -Hub $nuovo.AssignedHub -DeviceId $id -StateDir $nuovo.Dir
Show-HubAcceptance -Label 'certificato revocato, device riabilitato' -Hub $nuovo.AssignedHub -DeviceId $id -StateDir $prima.Dir


## Il device si registra di nuovo
Quello che ripristina l'accesso, ogni volta provato, è una **nuova registrazione a DPS**: il device ottiene un certificato nuovo e l'Hub torna ad accettare anche il certificato valido di prima. Il revocato resta rifiutato.


In [ ]:
$ripreso = Open-ScenarioDevice -Config $cfg -DeviceId $id -Notebook '13-device-riabilitato' -Fresh
Invoke-DeviceTick $ripreso
"certificato ottenuto: $((Get-DeviceCertStatus $ripreso).State)"
Show-HubAcceptance -Label 'certificato nuovo, dopo la nuova registrazione' -Hub $ripreso.AssignedHub -DeviceId $id -StateDir $ripreso.Dir
Show-HubAcceptance -Label 'certificato valido di prima, dopo la nuova registrazione' -Hub $ripreso.AssignedHub -DeviceId $id -StateDir $nuovo.Dir
Show-HubAcceptance -Label 'certificato revocato, dopo la nuova registrazione' -Hub $ripreso.AssignedHub -DeviceId $id -StateDir $prima.Dir


Da qui la regola per un device compromesso: **prima si disabilita** (ferma subito l'accesso, in meno di un secondo, e impedisce di ottenere un certificato nuovo), **poi si revoca** (i vecchi certificati non tornano validi alla riabilitazione: il revocato resta rifiutato anche dopo).
Alla riabilitazione, invece, **non c'è un tempo certo**: l'Hub può continuare a rifiutare anche un certificato valido per molti minuti. Si risolve facendo registrare di nuovo il device.


Prossimo notebook: **14**, gruppi, aggiornamenti e job.